# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [1]:
# imports
import os
import sys
from pathlib import Path

import requests
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

# Local helpers (notebook cwd may be repo root or this folder)
for _p in (Path.cwd(), Path.cwd() / "week1/community-contributions/jcburolleau"):
    if (_p / "openai_llm.py").is_file():
        sys.path.insert(0, str(_p.resolve()))
        break

import openai_llm
import ollama_llm


In [2]:
# constants

MODEL_GPT = "gpt-4o-mini"
OLLAMA_HOST = "http://localhost:11434"


In [3]:
# set up environment
load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

openai = OpenAI()
ollama_client = ollama_llm.make_ollama_client(OLLAMA_HOST)

if not api_key:
    print(
        "No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!"
    )


**Usage:** Run the setup cells once (through `sys_message`). Then run the cells below in order for each Q&A session.

In [4]:
sys_message = (
    "You need to behave like a complete expert teacher on explaining tecnical questions "
    "and explain them in a very simple and concise way, if the user is not asking a proper "
    "tecnical question, you need to ask for one."
)


### 1) Provider

OpenAI (cloud) or Ollama (local).


In [5]:
while True:
    provider = input("1 = OpenAI (cloud), 2 = Ollama (local). Your choice: ").strip()
    if provider in ("1", "2"):
        break
    print("Type 1 or 2.", flush=True)


Type 1 or 2.


### 2) Model

OpenAI always uses `MODEL_GPT`. For Ollama, the next cell shows models in the cell output **and repeats them inside the input prompt** so you can pick a number next to the cursor.


In [6]:
if provider == "1":
    client = openai
    model = MODEL_GPT
    print(f"Using OpenAI model: {model}", flush=True)
else:
    try:
        ollama_models = ollama_llm.fetch_installed_model_names(OLLAMA_HOST)
    except requests.RequestException as e:
        raise RuntimeError(
            f"Could not reach Ollama at {OLLAMA_HOST} ({e}). "
            "Is the app running or `ollama serve` active?"
        ) from e
    if not ollama_models:
        raise RuntimeError("No models found. Install one with e.g. `ollama pull llama3.2`.")

    menu_lines = "\n".join(f"  {i}. {name}" for i, name in enumerate(ollama_models, 1))
    display(Markdown("### Installed Ollama models\n\n" + "\n".join(f"{i}. `{n}`" for i, n in enumerate(ollama_models, 1))))
    sys.stdout.flush()

    while True:
        raw = input(
            "Installed Ollama models:\n"
            f"{menu_lines}\n\n"
            f"Model number [1-{len(ollama_models)}]: "
        ).strip()
        try:
            idx = int(raw)
        except ValueError:
            print("Enter a whole number.", flush=True)
            continue
        if 1 <= idx <= len(ollama_models):
            model = ollama_models[idx - 1]
            break
        print(f"Pick between 1 and {len(ollama_models)}.", flush=True)
    client = ollama_client
    print(f"Using Ollama model: {model}", flush=True)


Using OpenAI model: gpt-4o-mini


### 3) Question


In [7]:
question = input("Your technical question: ").strip()
messages = [
    {"role": "system", "content": sys_message},
    {"role": "user", "content": question},
]


### 4) Streamed answer


In [8]:
if provider == "1":
    stream = openai_llm.stream_chat_completions(client, model, messages)
else:
    stream = ollama_llm.stream_ollama_chat(client, model, messages)
reply = ""
out = display(Markdown(""), display_id=True)
for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        reply += delta
        update_display(Markdown(reply), display_id=out.display_id)
if not reply:
    print("No response from the model")


To define a variable in programming, you need to give it a name and specify its type. Here's a simple breakdown:

1. **Choose a Name**: This is how you will refer to the variable in your code. It should be meaningful and descriptive, like `age` or `totalScore`.

2. **Specify its Type**: This tells the program what kind of data the variable will hold, such as:
   - **Integer** (whole numbers, like 5 or -3)
   - **Float** (decimal numbers, like 3.14)
   - **String** (text, like "Hello")
   - **Boolean** (true or false)

3. **Assign a Value**: You give the variable an initial value. 

For example, in Python, you can define a variable like this:

```python
age = 25  # Here, 'age' is the variable name, and it's an integer.
```

If you have a specific programming language or context in mind, please let me know!